In [3]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID = "distilbert/distilbert-base-uncased"
DATASET = "rotten_tomatoes"
OUTPUT_DIR = "distilbert-rotten-tomatoes"
# ─────────────────────────────────────────────────────────────────────────────


# Model & tokenizer
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
dataset = load_dataset(DATASET)
dataset = dataset.map(lambda x: tokenizer(x["text"]), batched=True)

# Training
trainer = Trainer(
    model           = model,
    args            = TrainingArguments(
        output_dir                  = OUTPUT_DIR,
        learning_rate               = 2e-5,
        per_device_train_batch_size = 8,
        per_device_eval_batch_size  = 8,
        num_train_epochs            = 2,
        eval_strategy               = "epoch",
        push_to_hub                 = False,
    ),
    train_dataset    = dataset["train"],
    eval_dataset     = dataset["test"],
    processing_class = tokenizer,
    data_collator    = DataCollatorWithPadding(tokenizer=tokenizer),
)

trainer.train()


Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,0.388588,0.402693
2,0.275713,0.554139


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2134, training_loss=0.3387606316006061, metrics={'train_runtime': 37.6089, 'train_samples_per_second': 453.616, 'train_steps_per_second': 56.742, 'total_flos': 195974132394480.0, 'train_loss': 0.3387606316006061, 'epoch': 2.0})

In [6]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/themaximkol/distilbert-rotten-tomatoes/commit/4ae7951dc1d002f15ec6d9a828568c25f2885da4', commit_message='End of training', commit_description='', oid='4ae7951dc1d002f15ec6d9a828568c25f2885da4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/themaximkol/distilbert-rotten-tomatoes', endpoint='https://huggingface.co', repo_type='model', repo_id='themaximkol/distilbert-rotten-tomatoes'), pr_revision=None, pr_num=None)